# Consolidated vessel data preparation

## Purpose

This notebook prepares the harmonised vessel datasets consumed by the EFFKG
Wikibase import pipeline.

It integrates records from the European Fleet Register with country-specific
national fleet data, normalises heterogeneous source fields, reconciles vessel
records through the CFR identifier and exports a unified intermediate CSV
matching the input schema expected by `import_wikibase.ipynb`.

## Processing modes

The notebook supports two execution modes.

### `eu_only`

Processes European Fleet Register records without national enrichment.

### `national_eu`

Integrates national registry records with the corresponding European Fleet
Register records.

In `national_eu` mode:

1. all national registry records are retained;
2. national records are enriched with European Fleet Register values when the
   CFR identifiers match exactly;
3. European Fleet Register records whose CFR identifiers are not present in the
   national dataset are appended to the consolidated output;
4. national identifiers remain available as fallback identifiers for records
   without a usable CFR.

## Harmonisation behaviour

The notebook:

- aligns source-specific column names to a common vessel schema;
- preserves distinct values from multiple sources when they differ;
- combines main and subsidiary fishing-gear fields;
- maps European hull-material codes to controlled labels;
- extracts event-specific destruction and retirement dates;
- normalises registration-place names;
- repairs common text-encoding artefacts;
- preserves source-specific operational, technical and registry information.

The notebook does not write to Wikibase.

## Reconciliation statistics

In `national_eu` mode, the notebook also reports CFR-based reconciliation
statistics, including:

- source-record counts;
- records without valid CFR values;
- unique CFR counts per source;
- exact shared CFR matches;
- national-only CFR values;
- EUFR-only CFR values;
- overlap proportions.

These statistics are calculated independently of the generated consolidated
dataset and do not modify the consolidation workflow.

## Output

The generated CSV is written to:

`source_data/processed/consolidated_data_by_country/<input_mode>/`

The resulting file is consumed directly by:

`code/import_wikibase.ipynb`

# Setup and configuration

This section:

- locates the EFFKG repository root;
- defines repository-relative input and output paths;
- selects the country and consolidation mode;
- validates the required source files;
- creates the output directory;
- prints the active configuration.

The notebook uses only local source files distributed or prepared within the
EFFKG repository and does not require credentials or access to Wikibase.

In [ ]:
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import re


def find_repository_root(start: Optional[Path] = None) -> Path:
    """
    Locate the EFFKG repository root from the current working directory.

    The root is identified through the presence of the repository directories
    used by the processing pipeline.
    """
    current = (start or Path.cwd()).resolve()

    required_directories = {
        "code",
        "dataset",
        "schema",
        "source_data",
        "data_model",
        "validation",
    }

    for candidate in [current, *current.parents]:
        existing = {
            path.name
            for path in candidate.iterdir()
            if path.is_dir()
        }

        if required_directories.issubset(existing):
            return candidate

    raise RuntimeError(
        "EFFKG repository root could not be located. "
        "Run the notebook from within a cloned EFFKG repository."
    )


REPOSITORY_ROOT = find_repository_root()

COUNTRY = "ESP"
INPUT_MODE = "national_eu"

NATIONAL_PATH = (
    REPOSITORY_ROOT
    / "source_data"
    / "national_data_by_country"
    / f"national_{COUNTRY}_data.csv"
)

EU_PATH = (
    REPOSITORY_ROOT
    / "source_data"
    / "eufr_data_by_country"
    / f"eufr_data_{COUNTRY}.csv"
)

OUTPUT_PATH = (
    REPOSITORY_ROOT
    / "source_data"
    / "processed"
    / "consolidated_data_by_country"
    / INPUT_MODE
    / f"consolidated_data_{COUNTRY}.csv"
)

required_inputs = [EU_PATH]

if INPUT_MODE == "national_eu":
    required_inputs.append(NATIONAL_PATH)

missing_inputs = [
    path for path in required_inputs
    if not path.is_file()
]

if missing_inputs:
    missing_list = "\n".join(
        f"  - {path}" for path in missing_inputs
    )

    raise FileNotFoundError(
        "Required source files were not found:\n"
        f"{missing_list}\n"
        "See source_data/README.md for input requirements."
    )

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Input mode: {INPUT_MODE}")
print(f"Output: {OUTPUT_PATH}")

Repository root: C:\Users\profesor\Documents\EFFKG
Input mode: national_eu
Output: C:\Users\profesor\Documents\EFFKG\dataset\intermediate\consolidated_data_by_country\national_eu\consolidated_data_ESP.csv


In [ ]:


materiales_eu = {
    1: "Wood",
    2: "Metal",
    3: "Fiber/Plastic",
    4: "Other",
    5: "Unknown",
    6: "Polyester",
    7: "Aluminium"
}

eventos_eu = {
    "CEN": "Census",
    "CHA": "Change of Activity",
    "CST": "New Construction",
    "DES": "Destruction, wreck",
    "EXP": "Exportation, transfer",
    "IMP": "Importation, transfer",
    "MOD": "Modification",
    "RET": "Change of activity (exit)"
}

# Utilities

This section defines helper functions used to:

- clean scalar values;
- preserve distinct source values;
- select the first available value from equivalent columns;
- map European hull-material codes;
- combine fishing-gear fields;
- extract operational and event-specific information;
- normalize registration-place names;
- repair text-encoding artefacts;
- calculate CFR-based reconciliation statistics.

In [ ]:
"""Normalize CSV cell values and convert empty strings to None."""
def clean_value(v):
    if pd.isna(v):
        return None
    v = str(v).strip()
    return v if v != "" else None

"""Merge two values preserving uniqueness and returning pipe-separated output when needed."""
def merge_values(a, b):
    a = clean_value(a)
    b = clean_value(b)

    if a is None and b is None:
        return np.nan
    if a is None:
        return b
    if b is None:
        return a
    if a == b:
        return a

    values = []
    for v in [a, b]:
        if v not in values:
            values.append(v)
    return "|".join(values)

"""Merge an arbitrary number of values preserving unique non-empty entries."""
def merge_many(*values):
    cleaned = []
    for v in values:
        v = clean_value(v)
        if v is not None and v not in cleaned:
            cleaned.append(v)

    if not cleaned:
        return np.nan
    if len(cleaned) == 1:
        return cleaned[0]
    return "|".join(cleaned)

"""Return the first non-empty value from a list of candidate columns."""
def first_existing(row, *cols):
    for col in cols:
        if col in row.index:
            v = clean_value(row[col])
            if v is not None:
                return v
    return np.nan

"""Merge semantically equivalent source columns while preserving distinct values."""
def merge_equivalent_columns(row, *cols):
    values = []
    for col in cols:
        if col in row.index:
            v = clean_value(row[col])
            if v is not None and v not in values:
                values.append(v)

    if not values:
        return np.nan
    if len(values) == 1:
        return values[0]
    return "|".join(values)

"""Translate EU hull material numeric codes into normalized labels."""
def map_material_eu(v):
    v = clean_value(v)
    if v is None:
        return None
    try:
        return materiales_eu.get(int(v), v)
    except Exception:
        return v

"""Build a consolidated fishing gear field from multiple source columns."""
def build_fishing_gear(row):
    cols = [
        "Main fishing gear",
        "Subsidiary fishing gear 1",
        "Subsidiary fishing gear 2",
        "Subsidiary fishing gear 3",
        "Subsidiary fishing gear 4",
        "Subsidiary fishing gear 5"
    ]

    values = []
    for col in cols:
        if col in row.index:
            v = clean_value(row[col])
            if v is not None and v not in values:
                values.append(v)

    if not values:
        return np.nan
    return ", ".join(values)

"""Extract the national operating-area label from the source census field."""
def extract_operates_at(v):
    v = clean_value(v)
    if v is None:
        return np.nan

    if " EN " in v:
        part = v.split(" EN ", 1)[1].strip()
        if part.startswith("EL "):
            part = part[3:].strip()
        return part if part else np.nan

    return np.nan

"""Return the event date only when the requested event code matches."""
def event_date_if(row, code):
    event = clean_value(row.get("Event"))
    date = clean_value(row.get("Event Start Date"))

    if event == code and event in eventos_eu and date is not None:
        return date
    return np.nan

"""Repair common UTF-8 text decoded incorrectly as Latin-1 when possible."""
def fix_mojibake(v):
    v = clean_value(v)
    if v is None:
        return np.nan

    try:
        repaired = v.encode("latin1").decode("utf-8")
        return repaired
    except Exception:
        return v

"""Normalize Spanish registration place names for consistent matching."""
def normalize_registration_place_esp(value):
    if pd.isna(value):
        return value

    text = str(value).strip()
    if not text:
        return text

    # 1. Remove prefixes such as "34100 - "
    text = re.sub(r"^\s*\d+\s*-\s*", "", text)

    # 2. Remove any parenthesized clarification text (y el espacio previo)
    text = re.sub(r"\s*\(.*?\)", "", text)

    # 3. Keep only the first segment before commas
    text = text.split(",")[0].strip()

    # 4. Normalize repeated whitespace
    text = re.sub(r"\s+", " ", text)

    return text


In [ ]:
from pathlib import Path


def normalize_cfr_series(series):
    """
    Normalize CFR values for matching and statistics.

    Values used as placeholders for missing identifiers are converted to pd.NA.
    """
    missing_tokens = {
        "",
        "-",
        "--",
        "N/A",
        "NA",
        "NAN",
        "NONE",
        "NULL"
    }

    normalized = (
        series.astype("string")
        .str.strip()
        .str.upper()
    )

    return normalized.mask(normalized.isin(missing_tokens), pd.NA)


def calculate_vessel_reconciliation_statistics(
    national_df,
    eu_df,
    country
):
    """
    Calculate exact CFR-based reconciliation statistics for the national and
    European source files.

    The function reports source coverage and overlap only. It does not alter the
    input dataframes or the consolidated output.
    """
    nat_cfr = normalize_cfr_series(national_df["CFR"])
    eu_cfr = normalize_cfr_series(eu_df["CFR"])

    national_records = len(national_df)
    eufr_records = len(eu_df)

    national_missing_cfr = int(nat_cfr.isna().sum())
    eufr_missing_cfr = int(eu_cfr.isna().sum())

    national_cfr_set = set(nat_cfr.dropna())
    eufr_cfr_set = set(eu_cfr.dropna())

    shared_cfr = national_cfr_set & eufr_cfr_set
    national_only_cfr = national_cfr_set - eufr_cfr_set
    eufr_only_cfr = eufr_cfr_set - national_cfr_set
    union_cfr = national_cfr_set | eufr_cfr_set

    statistics = {
        "country": country,
        "national_source_records": national_records,
        "eufr_source_records": eufr_records,
        "national_records_without_valid_cfr": national_missing_cfr,
        "eufr_records_without_valid_cfr": eufr_missing_cfr,
        "national_unique_valid_cfr": len(national_cfr_set),
        "eufr_unique_valid_cfr": len(eufr_cfr_set),
        "shared_cfr_exact_matches": len(shared_cfr),
        "national_only_cfr": len(national_only_cfr),
        "eufr_only_cfr": len(eufr_only_cfr),
        "unique_cfr_union": len(union_cfr),
        "shared_cfr_over_union_pct": round(
            100 * len(shared_cfr) / len(union_cfr), 2
        ) if union_cfr else 0.0,
        "national_cfr_linked_to_eufr_pct": round(
            100 * len(shared_cfr) / len(national_cfr_set), 2
        ) if national_cfr_set else 0.0,
        "eufr_cfr_linked_to_national_pct": round(
            100 * len(shared_cfr) / len(eufr_cfr_set), 2
        ) if eufr_cfr_set else 0.0,
    }

    return statistics

In [ ]:
"""Ensure that a dataframe contains all required columns."""
def ensure_columns(df, columns, default=np.nan):
    for col in columns:
        if col not in df.columns:
            df[col] = default
    return df

"""
Load the configured vessel sources and build the combined source dataframe.

In `national_eu` mode, all national records are retained, matching European
records are joined by exact CFR equality, and EUFR-only records are appended.

In `eu_only` mode, the European Fleet Register input is returned directly.
"""
def prepare_dataframe(input_mode=INPUT_MODE, nat_path=NATIONAL_PATH, eu_path=EU_PATH):
    csv2 = pd.read_csv(eu_path, sep=";", dtype=str, encoding="utf-8")
    csv2.rename(columns={"CFR": "CFR"}, inplace=True)

    if input_mode == "national_eu":
        csv1 = pd.read_csv(nat_path, sep=";", dtype=str)
        csv1.rename(columns={"CFR": "CFR"}, inplace=True)

        # Retain every national registry record and enrich it with EUFR values
        # when an exact CFR match is available.
        df_main = csv1.merge(csv2, on="CFR", how="left", suffixes=("_csv1", "_csv2"))

        # Identify EUFR records whose CFR is not present in the national source.
        missing_eu = csv2[~csv2["CFR"].isin(csv1["CFR"])].copy()

        # Add empty national columns to keep schema compatibility
        national_only_cols = [col for col in csv1.columns if col != "CFR"]
        for col in national_only_cols:
            if col not in missing_eu.columns:
                missing_eu[col] = np.nan

        # Reorder columns to match the merged dataframe schema
        for col in df_main.columns:
            if col not in missing_eu.columns:
                missing_eu[col] = np.nan
        missing_eu = missing_eu[df_main.columns]

        # Build the complete consolidated population:
        # - all national source records;
        # - all EUFR-only records.
        df = pd.concat([df_main, missing_eu], ignore_index=True)

    elif input_mode == "eu_only":
        df = csv2.copy()

    else:
        raise ValueError("INPUT_MODE must be 'national_eu' o 'eu_only'")

    expected_cols = [
        "CFR",
        "MMSI",
        "Country of Registration",
        "Administración responsable del Registro",
        "Administration responsible of registration",
        "Alta en RGFP",
        "Registration on national registry",
        "Tonnage GT",
        "Other tonnage",
        "GTs",
        "Código",
        "Code",
        "LOA",
        "LBP",
        "Nombre",
        "Name",
        "Registration Number",
        "External marking",
        "Place of registration name",
        "Estado",
        "Status on national registry",
        "IRCS indicator",
        "IRCS_csv2",
        "IRCS",
        "VMS indicator",
        "ERS indicator",
        "AIS indicator",
        "UVI",
        "IMO",
        "Vessel Type",
        "Censo por modalidad",
        "Operates at",
        "Main fishing gear",
        "Subsidiary fishing gear 1",
        "Subsidiary fishing gear 2",
        "Subsidiary fishing gear 3",
        "Subsidiary fishing gear 4",
        "Subsidiary fishing gear 5",
        "Potencia",
        "Power of main engine",
        "Power of auxiliary engine",
        "Material del casco",
        "Hull material",
        "Image",
        "Date of entry into service",
        "Segment",
        "Year of construction",
        "Event",
        "Event Start Date"
    ]
    df = ensure_columns(df, expected_cols)

    df["IRCS_csv2"] = df.apply(lambda row: first_existing(row, "IRCS_csv2", "IRCS"), axis=1)

    return df

COUNTRY_CONFIG = {
    "ESP": {
            "Name of vessel": "Nombre",
            "Status on national registry": "Estado",
            "Administration responsible of registration": "Administración responsable del Registro",
            "Registration on national registry": "Alta en RGFP",
            "Code": "Código",
            "Operates at": "Censo por modalidad",
            "Power of main engine": "Potencia",
            "Hull material": "Material del casco",
            "Image": "Imagen",
            "Registration Place": "Puerto base"
    }
}

"""Resolve country-specific national column mappings."""
def resolve_national_property(prop):
    if INPUT_MODE == "national_eu":
        return COUNTRY_CONFIG[COUNTRY][prop]
    elif INPUT_MODE == "eu_only":
        return prop
    else:
        raise ValueError("INPUT_MODE must be 'national_eu' o 'eu_only'")


# Execution

This section builds the vessel table expected by the
Wikibase importer.

In [ ]:
df = prepare_dataframe()

registration_place_nat = df[resolve_national_property("Registration Place")].apply(
    normalize_registration_place_esp
)

csv3 = pd.DataFrame({
    "CFR": df["CFR"],
    "MSSI": df["MMSI"],
    "Country of registration": df["Country of Registration"],

    "Administration responsible of registration": df.apply(
        lambda row: first_existing(
            row,
            resolve_national_property("Administration responsible of registration"),
            "Administration responsible of registration"
        ),
        axis=1
    ),

    "Registration on national registry": df.apply(
        lambda row: first_existing(
            row,
            resolve_national_property("Registration on national registry"),
            "Registration on national registry"
        ),
        axis=1
    ),

    "GT Tonnage": df["Tonnage GT"],
    "Other tonnage": df["Other tonnage"],
    "GTs": df["GTs"],

    "Code": df.apply(
        lambda row: first_existing(
            row,
            resolve_national_property("Code"),
            "Code"
        ),
        axis=1
    ),

    "LOA": df["LOA"],
    "LBP": df["LBP"],

    "Name": df.apply(
        lambda row: first_existing(
            row,
            resolve_national_property("Name of vessel"),
            "Name of vessel"
        ),
        axis=1
    ),

    "Registration number": df["Registration Number"],
    "External marking": df["External marking"],
    "Registration Place": df["Place of registration name"].combine_first(registration_place_nat),
    "Status on national registry": df.apply(
        lambda row: first_existing(
            row,
            resolve_national_property("Status on national registry"),
            "Status on national registry"
        ),
        axis=1
    ),

    "Radio": df["IRCS indicator"],
    "IRCS": df["IRCS_csv2"],
    "VMS": df["VMS indicator"],
    "ERS": df["ERS indicator"],
    "AIS": df["AIS indicator"],
    "UVI": df["UVI"],
    "IMO": df["IMO"],
    "Vessel Type": df["Vessel Type"],

    "Operates at": df.apply(
        lambda row: first_existing(
            row,
            resolve_national_property("Operates at"),
            "Operates at"
        ),
        axis=1
    ).apply(extract_operates_at),

    "Fishing gear": df.apply(build_fishing_gear, axis=1),

    "Main engine": df.apply(
        lambda row: merge_values(
            row.get(resolve_national_property("Power of main engine")),
            row.get("Power of main engine")
        ),
        axis=1
    ),

    "Power of auxiliary engine": df["Power of auxiliary engine"],

    "Hull material": df.apply(
        lambda row: (
            map_material_eu(row.get("Hull material"))
            if INPUT_MODE == "eu_only"
            else merge_values(
                row.get(resolve_national_property("Hull material")),
                map_material_eu(row.get("Hull material"))
            )
        ),
        axis=1
    ),

    "Image": df.apply(
        lambda row: first_existing(
            row,
            resolve_national_property("Image"),
            "Image"
        ),
        axis=1
    ),

    "Date of entry into service": df["Date of entry into service"],
    "Segment": df["Segment"],
    "Year of construction": df["Year of construction"],
    "Year of destruction": df.apply(lambda row: event_date_if(row, "DES"), axis=1),
    "Year of retirement": df.apply(lambda row: event_date_if(row, "RET"), axis=1)
})

text_columns_to_fix = [
    "Name",
    "Registration Place"
]

for col in text_columns_to_fix:
    if col in csv3.columns:
        csv3[col] = csv3[col].apply(fix_mojibake)

csv3.to_csv(OUTPUT_PATH, index=False, sep=";")
print(f"CSV generated with INPUT_MODE={INPUT_MODE}")

if INPUT_MODE == "national_eu":
    nat_stats_df = pd.read_csv(NATIONAL_PATH, sep=";", dtype=str)
    eu_stats_df = pd.read_csv(EU_PATH, sep=";", dtype=str)

    reconciliation_stats = calculate_vessel_reconciliation_statistics(
        national_df=nat_stats_df,
        eu_df=eu_stats_df,
        country=COUNTRY
    )

    print("\nVESSEL RECONCILIATION STATISTICS")
    print("--------------------------------")
    for key, value in reconciliation_stats.items():
        print(f"{key}: {value}")

CSV generated with INPUT_MODE=national_eu

VESSEL RECONCILIATION STATISTICS
--------------------------------
country: ESP
national_source_records: 30535
eufr_source_records: 27507
national_records_without_valid_cfr: 2889
eufr_records_without_valid_cfr: 0
national_unique_valid_cfr: 27645
eufr_unique_valid_cfr: 27507
shared_cfr_exact_matches: 27379
national_only_cfr: 266
eufr_only_cfr: 128
unique_cfr_union: 27773
shared_cfr_over_union_pct: 98.58
national_cfr_linked_to_eufr_pct: 99.04
eufr_cfr_linked_to_national_pct: 99.53
